## Pull AQI data for nfl gamedays

In [6]:
import pandas as pd

combined = pd.read_csv('data/all_pollutants_daily_clean_2023-2025.csv')
combined['Date Local'] = pd.to_datetime(combined['Date Local'])

# Game dates array
game_dates = pd.to_datetime([
    "2023-09-11", "2023-09-24", "2023-10-01", "2023-10-15", "2023-11-06", 
    "2023-11-24", "2023-12-03", "2023-12-10", "2023-12-24", "2024-09-19", 
    "2024-09-29", "2024-10-14", "2024-10-31", "2024-11-17", "2024-12-01", 
    "2024-12-22", "2025-01-05", "2025-09-07", "2025-09-14", "2025-10-05", 
    "2025-10-19", "2025-11-09", "2025-11-30", "2025-12-07", "2025-12-28", 
    "2023-09-10", "2023-10-02", "2023-10-22", "2023-10-29", "2023-11-26", 
    "2023-12-11", "2023-12-31", "2024-01-07", "2024-09-08", "2024-09-26", 
    "2024-10-13", "2024-10-20", "2024-11-03", "2024-11-24", "2024-12-08", 
    "2024-12-15", "2024-12-28", "2025-09-21", "2025-09-28", "2025-10-09", 
    "2025-10-26", "2025-11-02", "2025-11-16", "2025-12-14", "2025-12-21" 
])

nfl_df = combined[
    combined['Date Local'].isin(game_dates)
].copy()

nfl_df['overall_aqi'] = nfl_df.groupby(
    ['Local Site Name', 'Date Local']
)['daily_max_aqi'].transform('max')

def get_main_pollutant(group):
    max_aqi = group['daily_max_aqi'].max()
    top = group[group['daily_max_aqi'] == max_aqi]['pollutant'].tolist()
    return ' / '.join(sorted(top)) if len(top) > 1 else top[0]

main_poll = nfl_df.groupby(
    ['Local Site Name', 'Date Local']
).apply(get_main_pollutant).reset_index()
main_poll.columns = ['Local Site Name', 'Date Local', 'main_pollutant']

nfl_df = nfl_df.drop(columns='main_pollutant', errors='ignore').merge(
    main_poll, on=['Local Site Name', 'Date Local']
)

nfl_df.to_csv('NFL_gameday_airquality.csv', index=False)

# Comparison Group

In [ ]:
"""
Builds four comparison groups from the daily pollutant data for the
NJ DEP World Cup air quality analysis.

Groups produced:
  1. nfl_gameday_aqi.csv       — AQI on NFL game days (34 matched dates)
  2. nfl_control_aqi.csv       — Same day-of-week, one week prior, non-game
  3. summer_baseline_aqi.csv   — Jun–Aug 2023/2024/2025, no game days
  4. winter_baseline_aqi.csv   — Nov–Dec 2023/2024/2025, no game days

Each output file has the same schema so they can be directly compared.
A summary CSV is also saved with one row per group per county per pollutant.

Input:  all_pollutants_daily_clean_2023-2025.csv
Output: data/nfl_gameday_aqi.csv
        data/nfl_control_aqi.csv
        data/summer_baseline_aqi.csv
        data/winter_baseline_aqi.csv
        data/comparison_summary.csv
"""

import os
import pandas as pd

# CONFIG
INPUT_FILE  = "data/all_pollutants_daily_clean_2023-2025.csv"   # update path if needed
OUTPUT_DIR  = "data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# NFL GAME DATES 
# 50 total — Giants and Jets home games at MetLife, 2023–2025 seasons.
# 16 fall outside the dataset's Aug 2025 cutoff; those will simply return
# no rows and are excluded automatically.
game_dates = pd.to_datetime([
    "2023-09-11", "2023-09-24", "2023-10-01", "2023-10-15", "2023-11-06",
    "2023-11-24", "2023-12-03", "2023-12-10", "2023-12-24", "2024-09-19",
    "2024-09-29", "2024-10-14", "2024-10-31", "2024-11-17", "2024-12-01",
    "2024-12-22", "2025-01-05", "2025-09-07", "2025-09-14", "2025-10-05",
    "2025-10-19", "2025-11-09", "2025-11-30", "2025-12-07", "2025-12-28",
    "2023-09-10", "2023-10-02", "2023-10-22", "2023-10-29", "2023-11-26",
    "2023-12-11", "2023-12-31", "2024-01-07", "2024-09-08", "2024-09-26",
    "2024-10-13", "2024-10-20", "2024-11-03", "2024-11-24", "2024-12-08",
    "2024-12-15", "2024-12-28", "2025-09-21", "2025-09-28", "2025-10-09",
    "2025-10-26", "2025-11-02", "2025-11-16", "2025-12-14", "2025-12-21",
])

game_dates_set = set(game_dates.normalize())

# NON-GAMEDAY CONTROL: same day-of-week, exactly one week prior
# For each game date, the control is that same weekday 7 days earlier.
# Any control date that is itself a game day is skipped — in practice this
# doesn't occur in this dataset but the check is here for correctness.
control_dates = pd.to_datetime([
    d - pd.Timedelta(days=7) for d in game_dates
    if (d - pd.Timedelta(days=7)) not in game_dates_set
])
control_dates_set = set(control_dates.normalize())

# LOAD DATA
print("Loading data...")
df = pd.read_csv(INPUT_FILE, low_memory=False)
df['Date Local'] = pd.to_datetime(df['Date Local'])
df['month']      = df['Date Local'].dt.month
df['dow']        = df['Date Local'].dt.dayofweek   # 0=Mon … 6=Sun

print(f"  Rows: {len(df):,}  |  Stations: {df['Local Site Name'].nunique()}  "
      f"|  Date range: {df['Date Local'].min().date()} → {df['Date Local'].max().date()}")

# GROUP 1: NFL GAME DAYS 
nfl_gameday = df[df['Date Local'].isin(game_dates_set)].copy()
nfl_gameday['group'] = 'NFL Game Day'

matched_dates = nfl_gameday['Date Local'].nunique()
print(f"\nGroup 1 — NFL game days: {matched_dates} dates matched "
      f"({len(game_dates) - matched_dates} fall outside dataset window)")

# GROUP 2: CONTROL — SAME DOW, ONE WEEK PRIOR
nfl_control = df[df['Date Local'].isin(control_dates_set)].copy()
nfl_control['group'] = 'NFL Control'

# Verify pairing: show game date → control date for the first 10
print(f"\nGroup 2 — Control dates: {nfl_control['Date Local'].nunique()} dates matched")
print("  Sample pairings (game → control):")
paired = sorted(zip(game_dates, [d - pd.Timedelta(days=7) for d in game_dates]))
for game, ctrl in paired[:8]:
    in_data  = "✓" if game.normalize() in game_dates_set and \
               df['Date Local'].isin([game]).any() else "✗ not in dataset"
    ctrl_str = ctrl.strftime("%Y-%m-%d (%a)")
    game_str = game.strftime("%Y-%m-%d (%a)")
    print(f"    {game_str} → {ctrl_str}  [{in_data}]")

# GROUP 3: SUMMER BASELINE — JUN–AUG, NO GAME DAYS
summer_mask = (
    df['month'].isin([6, 7, 8]) &
    ~df['Date Local'].isin(game_dates_set)
)
summer_baseline = df[summer_mask].copy()
summer_baseline['group'] = 'Summer Baseline'

print(f"\nGroup 3 — Summer baseline (Jun–Aug, no game days): "
      f"{summer_baseline['Date Local'].nunique()} unique dates")
print("  Year breakdown:")
print(summer_baseline.groupby(summer_baseline['Date Local'].dt.year)['Date Local']
      .nunique().rename('unique_dates').to_string())

# GROUP 4: WINTER BASELINE — NOV–DEC, NO GAME DAYS
winter_mask = (
    df['month'].isin([11, 12]) &
    ~df['Date Local'].isin(game_dates_set)
)
winter_baseline = df[winter_mask].copy()
winter_baseline['group'] = 'Winter Baseline'

print(f"\nGroup 4 — Winter baseline (Nov–Dec, no game days): "
      f"{winter_baseline['Date Local'].nunique()} unique dates")
print("  Year breakdown:")
print(winter_baseline.groupby(winter_baseline['Date Local'].dt.year)['Date Local']
      .nunique().rename('unique_dates').to_string())

# ── SAVE INDIVIDUAL GROUP FILES 
output_cols = [
    'Local Site Name', 'County Name', 'Latitude', 'Longitude',
    'Date Local', 'Year', 'pollutant',
    'daily_mean_reading', 'daily_max_aqi', 'daily_1st_max',
    'group'
]

files = {
    'nfl_gameday_aqi.csv':    nfl_gameday,
    'nfl_control_aqi.csv':    nfl_control,
    'summer_baseline_aqi.csv': summer_baseline,
    'winter_baseline_aqi.csv': winter_baseline,
}

print("\nSaving group files...")
for fname, gdf in files.items():
    path = os.path.join(OUTPUT_DIR, fname)
    gdf[output_cols].to_csv(path, index=False)
    print(f"  ✓ {fname}  ({len(gdf):,} rows, {gdf['Date Local'].nunique()} dates)")

# ── COMPARISON SUMMARY ────────────────────────────────────────────────────────
# One row per group × county × pollutant
# avg_mean_reading: mean of daily_mean_reading across all dates in group
# avg_aqi:         mean of daily_max_aqi
# peak_reading:    single highest daily_1st_max in the group
# n_days:          number of unique calendar days with data

print("\nBuilding comparison summary...")
all_groups = pd.concat([nfl_gameday, nfl_control, summer_baseline, winter_baseline],
                       ignore_index=True)

summary = (
    all_groups
    .groupby(['group', 'County Name', 'pollutant'])
    .agg(
        avg_mean_reading = ('daily_mean_reading', 'mean'),
        avg_aqi          = ('daily_max_aqi',      'mean'),
        peak_reading     = ('daily_1st_max',      'max'),
        n_days           = ('Date Local',         'nunique'),
    )
    .round(3)
    .reset_index()
)

# Add a statewide aggregate row (all counties averaged)
statewide = (
    all_groups
    .groupby(['group', 'pollutant'])
    .agg(
        avg_mean_reading = ('daily_mean_reading', 'mean'),
        avg_aqi          = ('daily_max_aqi',      'mean'),
        peak_reading     = ('daily_1st_max',      'max'),
        n_days           = ('Date Local',         'nunique'),
    )
    .round(3)
    .reset_index()
)
statewide.insert(1, 'County Name', 'STATEWIDE')

summary = pd.concat([summary, statewide], ignore_index=True)

summary_path = os.path.join(OUTPUT_DIR, 'comparison_summary.csv')
summary.to_csv(summary_path, index=False)
print(f"  ✓ comparison_summary.csv  ({len(summary):,} rows)")

# ── QUICK FINDINGS PREVIEW ────────────────────────────────────────────────────
print("\n" + "="*60)
print("STATEWIDE AVERAGES BY GROUP (avg AQI)")
print("="*60)
pivot = (
    statewide
    .pivot(index='pollutant', columns='group', values='avg_aqi')
    [['NFL Game Day', 'NFL Control', 'Summer Baseline', 'Winter Baseline']]
    .round(2)
)
print(pivot.to_string())

print("\n" + "="*60)
print("CORRIDOR COUNTIES (Bergen, Hudson, Passaic) — avg AQI by group")
print("="*60)
corridor = all_groups[all_groups['County Name'].isin(['Bergen','Hudson','Passaic'])]
corridor_summary = (
    corridor
    .groupby(['group', 'pollutant'])['daily_max_aqi']
    .mean()
    .unstack('group')
    [['NFL Game Day', 'NFL Control', 'Summer Baseline', 'Winter Baseline']]
    .round(2)
)
print(corridor_summary.to_string())

print("\n" + "="*60)
print("SHORE COUNTIES (Atlantic, Monmouth, Ocean) — avg AQI by group")
print("="*60)
shore = all_groups[all_groups['County Name'].isin(['Atlantic','Ocean'])]
# Note: Monmouth has no pollutant data — only ozone, no PM2.5 or NO2
shore_summary = (
    shore
    .groupby(['group', 'pollutant'])['daily_max_aqi']
    .mean()
    .unstack('group')
    [['NFL Game Day', 'NFL Control', 'Summer Baseline', 'Winter Baseline']]
    .round(2)
)
print(shore_summary.to_string())

print("\nDone. All files saved to:", OUTPUT_DIR)

Loading data...
  Rows: 36,234  |  Stations: 28  |  Date range: 2023-01-01 → 2025-08-31

Group 1 — NFL game days: 34 dates matched (16 fall outside dataset window)

Group 2 — Control dates: 21 dates matched
  Sample pairings (game → control):
    2023-09-10 (Sun) → 2023-09-03 (Sun)  [✓]
    2023-09-11 (Mon) → 2023-09-04 (Mon)  [✓]
    2023-09-24 (Sun) → 2023-09-17 (Sun)  [✓]
    2023-10-01 (Sun) → 2023-09-24 (Sun)  [✓]
    2023-10-02 (Mon) → 2023-09-25 (Mon)  [✓]
    2023-10-15 (Sun) → 2023-10-08 (Sun)  [✓]
    2023-10-22 (Sun) → 2023-10-15 (Sun)  [✓]
    2023-10-29 (Sun) → 2023-10-22 (Sun)  [✓]

Group 3 — Summer baseline (Jun–Aug, no game days): 272 unique dates
  Year breakdown:
Date Local
2023    92
2024    92
2025    88

Group 4 — Winter baseline (Nov–Dec, no game days): 106 unique dates
  Year breakdown:
Date Local
2023    53
2024    53

Saving group files...
  ✓ nfl_gameday_aqi.csv  (1,325 rows, 34 dates)
  ✓ nfl_control_aqi.csv  (812 rows, 21 dates)
  ✓ summer_baseline_aqi.csv  

## Traffic data for non-gameday

In [ ]:
"""
Pulls hourly traffic volume data from FHWA TMAS VOL files for each of the
34 confirmed NFL game dates (2023–2024 seasons, Giants and Jets home games
at MetLife Stadium). Computes a daily total vehicle count per station per
game date by summing all hours, lanes, and directions.

Inputs:
  data/Traffic/NJ_[MONTH]_[YEAR]__TMAS_.VOL  — 12 monthly traffic count files
                                            covering Sep–Jan for 2023/24
                                            and 2024/25 seasons
  data/Traffic/NJ_2025_TMAS_.STA                — station metadata file with GPS
                                            coordinates and location names

Output:
  data/gameday_traffic.csv               — one row per station per game date
                                           with daily vehicle totals and
                                           station metadata attached
"""

import os
import pandas as pd

# CONFIG
VOL_DIR    = "data/Traffic"
STA_FILE   = "data/Traffic/NJ_2025_(TMAS).STA"
OUTPUT_DIR = "data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "gameday_traffic.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# NFL GAME DATES — 34 confirmed home game dates at MetLife Stadium
# Covers 2023 Giants/Jets season and 2024 Giants/Jets season
# 2025 season dates excluded — fall outside the VOL file coverage window
game_dates = pd.to_datetime([
    "2023-09-10", "2023-09-11", "2023-09-24",
    "2023-10-01", "2023-10-02", "2023-10-15", "2023-10-22", "2023-10-29",
    "2023-11-06", "2023-11-24", "2023-11-26",
    "2023-12-03", "2023-12-10", "2023-12-11", "2023-12-24", "2023-12-31",
    "2024-01-07",
    "2024-09-08", "2024-09-19", "2024-09-26", "2024-09-29",
    "2024-10-13", "2024-10-14", "2024-10-20", "2024-10-31",
    "2024-11-03", "2024-11-17", "2024-11-24",
    "2024-12-01", "2024-12-08", "2024-12-15", "2024-12-22", "2024-12-28",
    "2025-01-05",
])

# Map each game date to the VOL filename it lives in
# Key: (month, year) → filename as stored in VOL_DIR
VOL_FILE_MAP = {
    (9,  2023): "NJ_SEP_2023 (TMAS).VOL",
    (10, 2023): "NJ_OCT_2023 (TMAS).VOL",
    (11, 2023): "NJ_NOV_2023 (TMAS).VOL",
    (12, 2023): "NJ_DEC_2023 (TMAS).VOL",
    (1,  2024): "NJ_JAN_2024 (TMAS).VOL",
    (9,  2024): "NJ_SEP_2024 (TMAS).VOL",
    (10, 2024): "NJ_OCT_2024 (TMAS).VOL",
    (11, 2024): "NJ_NOV_2024 (TMAS).VOL",
    (12, 2024): "NJ_DEC_2024 (TMAS).VOL",
    (1,  2025): "NJ_Jan_2025 (TMAS).VOL",
}

# Hour columns present in every VOL row — summed for daily total
HOUR_COLS = [f"hour_{str(h).zfill(2)}" for h in range(24)]

# LOAD STATION METADATA
# Deduplicate on station_id — multiple rows exist per station (one per lane/direction)
# Keep only the first occurrence to get a single set of coordinates and location name
print("Loading station metadata...")
sta = pd.read_csv(STA_FILE, sep="|", low_memory=False)
sta.columns = sta.columns.str.lower().str.strip()
sta = sta.drop_duplicates(subset="station_id").reset_index(drop=True)
sta = sta[["station_id", "latitude", "longitude", "station_location", "county_code"]].copy()
sta["station_id"] = sta["station_id"].astype(str).str.strip()
print(f"  {len(sta):,} unique stations loaded from STA file")

# IDENTIFY WHICH VOL FILES ARE NEEDED
# Group game dates by (month, year) so each VOL file is only loaded once
dates_by_vol = {}
for gd in game_dates:
    key = (gd.month, gd.year)
    dates_by_vol.setdefault(key, []).append(gd)

print(f"\nGame dates span {len(dates_by_vol)} VOL files:")
for key, dates in sorted(dates_by_vol.items()):
    fname = VOL_FILE_MAP[key]
    print(f"  {fname}  →  {len(dates)} game date(s): {[d.date() for d in dates]}")

# PULL TRAFFIC DATA FOR EACH VOL FILE
# For each file, filter to the game dates that fall within it,
# sum all hour columns across all lanes and directions per station per day,
# then append to the master list
all_chunks = []

for key, dates in sorted(dates_by_vol.items()):
    fname = VOL_FILE_MAP[key]
    fpath = os.path.join(VOL_DIR, fname)

    print(f"\nLoading {fname}...")
    vol = pd.read_csv(fpath, sep="|", low_memory=False)
    vol.columns = vol.columns.str.lower().str.strip()

    # Build a calendar date column from year/month/day fields
    vol["date"] = pd.to_datetime({
        "year":  vol["year_record"],
        "month": vol["month_record"],
        "day":   vol["day_record"]
    })

    # Filter to only the game dates this VOL file covers
    target_dates = pd.to_datetime(dates)
    vol_game = vol[vol["date"].isin(target_dates)].copy()
    print(f"  Rows matching game dates: {len(vol_game):,}")

    if vol_game.empty:
        print("  WARNING: no matching rows found — skipping")
        continue

    # Convert hour columns to numeric, coercing any non-numeric values to NaN
    for col in HOUR_COLS:
        vol_game[col] = pd.to_numeric(vol_game[col], errors="coerce")

    # Sum all hours per row, then aggregate across lanes and directions
    # Result: one daily total per station per game date
    vol_game["daily_total"] = vol_game[HOUR_COLS].sum(axis=1)
    vol_game["station_id"]  = vol_game["station_id"].astype(str).str.strip()

    daily = (
        vol_game
        .groupby(["station_id", "date"], as_index=False)
        .agg(daily_vehicle_count=("daily_total", "sum"))
    )

    all_chunks.append(daily)
    print(f"  Stations with data: {daily['station_id'].nunique():,}  |  "
          f"Rows produced: {len(daily):,}")

# COMBINE ALL CHUNKS
print("\nCombining all game date traffic chunks...")
traffic = pd.concat(all_chunks, ignore_index=True)
print(f"  Total rows before station join: {len(traffic):,}")

# ATTACH STATION METADATA
# Left join — keeps all traffic rows; stations missing from STA get NaN coords
traffic = traffic.merge(sta, on="station_id", how="left")

missing_meta = traffic["latitude"].isna().sum()
if missing_meta > 0:
    print(f"  WARNING: {missing_meta} rows have no station metadata (station not in STA file)")

# FINAL COLUMN ORDER AND SORT
traffic = traffic.rename(columns={"date": "date_local"})
traffic = traffic[[
    "station_id", "station_location", "county_code",
    "latitude", "longitude",
    "date_local", "daily_vehicle_count"
]].sort_values(["date_local", "station_id"]).reset_index(drop=True)

# SAVE
traffic.to_csv(OUTPUT_FILE, index=False)
print(f"\n✓ Saved: {OUTPUT_FILE}")
print(f"  Rows:              {len(traffic):,}")
print(f"  Unique stations:   {traffic['station_id'].nunique():,}")
print(f"  Unique game dates: {traffic['date_local'].nunique():,}")
print(f"  Date range:        {traffic['date_local'].min().date()} → "
      f"{traffic['date_local'].max().date()}")

Loading station metadata...
  505 unique stations loaded from STA file

Game dates span 10 VOL files:
  NJ_JAN_2024 (TMAS).VOL  →  1 game date(s): [datetime.date(2024, 1, 7)]
  NJ_Jan_2025 (TMAS).VOL  →  1 game date(s): [datetime.date(2025, 1, 5)]
  NJ_SEP_2023 (TMAS).VOL  →  3 game date(s): [datetime.date(2023, 9, 10), datetime.date(2023, 9, 11), datetime.date(2023, 9, 24)]
  NJ_SEP_2024 (TMAS).VOL  →  4 game date(s): [datetime.date(2024, 9, 8), datetime.date(2024, 9, 19), datetime.date(2024, 9, 26), datetime.date(2024, 9, 29)]
  NJ_OCT_2023 (TMAS).VOL  →  5 game date(s): [datetime.date(2023, 10, 1), datetime.date(2023, 10, 2), datetime.date(2023, 10, 15), datetime.date(2023, 10, 22), datetime.date(2023, 10, 29)]
  NJ_OCT_2024 (TMAS).VOL  →  4 game date(s): [datetime.date(2024, 10, 13), datetime.date(2024, 10, 14), datetime.date(2024, 10, 20), datetime.date(2024, 10, 31)]
  NJ_NOV_2023 (TMAS).VOL  →  3 game date(s): [datetime.date(2023, 11, 6), datetime.date(2023, 11, 24), datetime.da